In [1]:
import sys
from pathlib import Path

project_root = Path().cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root = Path().cwd().parent

from wand_sim.engine.spells_db import SPELLS
from wand_sim.engine.spell import SpellType

print(f"总法术数: {len(SPELLS)}")
for s in SPELLS.values():
    print(f"  {s.id:20s} | {s.type.value:10s} | {s.name_zh}")

总法术数: 10
  LIGHT_BULLET         | projectile | 火花弹
  BULLET               | projectile | 魔法箭
  SLOW_BULLET          | projectile | 能量球
  RUBBER_BALL          | projectile | 弹跳爆发
  CHAINSAW             | projectile | 链锯
  DAMAGE               | modifier   | 伤害增强
  SPEED                | modifier   | 加速
  MANA_REDUCE          | modifier   | 额外法力
  BURST_2              | multicast  | 二重施法
  BURST_3              | multicast  | 三重施法


In [2]:
from wand_sim.engine.wand import WandStats, WandState

wand = WandStats(mana_max=200, mana_charge_speed=30)
state = WandState(stats=wand)
print(f"初始法力: {state.current_mana}")

ok = state.consume_mana(50)
print(f"消耗 50: {'成功' if ok else '失败'}, 剩余: {state.current_mana}")

state.regen_mana(2.0)
print(f"回复 2 秒: {state.current_mana:.0f}")

初始法力: 200
消耗 50: 成功, 剩余: 150
回复 2 秒: 200


## 模拟器验证

### 1.[火花弹]

In [3]:
from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

r = simulate(["spark_bolt"], wand, TargetInfo(distance_px=300))
print(f"DPS:{r.dps:.1f}")
print(f"法力耗尽时间：{r.mana_exhaustion_time:.1f}s（")
print(f"每秒发射：{r.avg_projectiles_per_second:.1f}")
print(f"命中率：{r.hit_rate:.1%}")

DPS:2.1
法力耗尽时间：10.5s（10=未耗尽）
每秒发射：1.8
命中率：32.6%


### 2.[二重施法, 火花弹, 火花弹]

In [10]:
from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

r = simulate(["double_spell", "spark_bolt", "spark_bolt"], wand, TargetInfo(distance_px=300))
print(f"DPS:{r.dps:.1f}")
print(f"法力耗尽时间：{r.mana_exhaustion_time:.1f}s")
print(f"每秒发射：{r.avg_projectiles_per_second:.1f}")
print(f"暴击占比：{r.crit_ratio:.1%}")

DPS:4.7
法力耗尽时间：3.4s
每秒发射：4.0
暴击占比：16.7%


### 3.[二重施法, 链锯, 火花弹]

In [18]:
from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

r = simulate(["double_spell", "chainsaw", "spark_bolt"], wand, TargetInfo(distance_px=300))
print(f"DPS:{r.dps:.1f}")
print(f"法力耗尽时间：{r.mana_exhaustion_time:.1f}s")
print(f"每秒发射：{r.avg_projectiles_per_second:.1f}")
print(f"命中率：{r.hit_rate:.1%}")

DPS:11.6
法力耗尽时间：5.6s
每秒发射：6.8
命中率：24.8%


### 4.对比：[火花弹] vs [伤害增强, 火花弹]

In [3]:
from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

r1 = simulate(["spark_bolt"], wand)
r2 = simulate(["damage_plus", "spark_bolt"], wand)
print(f"裸 DPS: {r1.dps:.1f}")
print(f"+Damage: {r2.dps:.1f}")

裸 DPS: 2.1
+Damage: 9.2


In [4]:
from optimizer.fitness import fitness

wand = WandStats(
    shuffle=False, cast_delay=0.17, recharge_time=0.33, mana_max=200, mana_charge_speed=30, capacity=8, spread=8,
)

res = fitness(["double_spell", "spark_bolt", "spark_bolt"], wand)
print(f"得分：{res}")
res = fitness(["spark_bolt"], wand)
print(f"得分：{res}")

得分：4.264675947739426
得分：2.1323379738697117


In [5]:
from wand_sim.engine.wand import WandStats
from wand_sim.engine.simulator import simulate, TargetInfo

wand = WandStats()
seq = ["double_spell", "spark_bolt", "spark_bolt"]

%timeit simulate(seq, wand)


103 μs ± 1.04 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
